In [2]:
# The first step is to load your dataset
import pandas as pd
df = pd.read_csv('GRN_experiment_M2_removal.csv')

In [6]:
# Secondly we are making profile report of this dataset to get detailed insight of data
from ydata_profiling import ProfileReport
inputProfileReport = ProfileReport(df)
inputProfileReport.to_file('inputProfileReport')

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  6.60it/s]
c:\Users\bhumi\AppData\Local\Programs\Python\Python311\Lib\site-packages\ydata_profiling\profile_report.py:386: UserWarning: Extension  not supported. For now we assume .html was intended. To remove this warning, please use .html or .json.
  warnings.warn(
Export report to file: 100%|██████████| 1/1 [00:00<?, ?it/s]


In [ ]:
# Above ProfileReportOfInputData is enough for understanding the data but we are going to manually
# Que1 : How big is the data?
df.shape # (6171, 7) It's not very big, so we can use whole dataset at a time

(6171, 7)

In [33]:
# Que2 : How does the data look like?
df.sample(5) # We can use df.head() but the df.sample() gives you random samples of dataset

,time,x,y,S1,S2,S3,S4
2388,1.52,0.1,0.8,0.225169,-0.304387,0.252300,-0.172561
1213,0.80,0.3,0.0,0.137543,-0.185211,0.152371,-0.102628
96,0.00,0.8,0.8,0.062571,-0.054376,0.028097,0.011421
2806,1.84,0.1,0.2,0.224838,-0.303961,0.252126,-0.173655
4123,2.72,0.9,0.0,0.030376,-0.041087,0.033959,-0.023707


In [34]:
# Que3 : What is the data type of columns?
df.info() # In our case all are numerical (float64) and using 337.6 KB of RAM

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6171 entries, 0 to 6170
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   time    6171 non-null   float64
 1   x       6171 non-null   float64
 2   y       6171 non-null   float64
 3   S1      6171 non-null   float64
 4   S2      6171 non-null   float64
 5   S3      6171 non-null   float64
 6   S4      6171 non-null   float64
dtypes: float64(7)
memory usage: 337.6 KB


In [35]:
# Que4 : Are there any missing values?
df.isnull().sum() # In our case there is no missing value in any column

time    0
x       0
y       0
S1      0
S2      0
S3      0
S4      0
dtype: int64

In [36]:
# Que5 : How does the data look mathematically?
df.describe() # We get the insight of dataset that what is the mean, standard deviation, min-max values, inter-quartile range, outlier if present etc...
# There are 7 columns in our dataset:
# column 'time' representing the time interval from starttime when the gene expression was measured at every time interval 0.08 sec
# columns 'x' and 'y' both are denoting coordinates value between 0 and 1 with the constant step value 0.1
# columns 'S1', 'S2', 'S3' and 'S4' are the value of gene expression recorded after every 0.08 sec in X-Y plan after removal of M2 morphogen


,time,x,y,S1,S2,S3,S4
count,6171.000000,6171.000000,6171.000000,6171.000000,6171.000000,6171.000000,6171.000000
mean,2.000000,0.500000,0.500000,0.123802,-0.153507,0.127443,-0.060514
std,1.177664,0.316253,0.316253,0.086254,0.111641,0.093761,0.098068
min,0.000000,0.000000,0.000000,0.023639,-0.392031,-0.155838,-0.225182
25%,0.960000,0.200000,0.200000,0.047372,-0.235732,0.045080,-0.120588
50%,2.000000,0.500000,0.500000,0.106192,-0.114657,0.101409,-0.046039
75%,3.040000,0.800000,0.800000,0.178694,-0.055650,0.195285,-0.018503
max,4.000000,1.000000,1.000000,0.462145,-0.031674,0.323427,0.609086


In [37]:
# Que6 : Are there duplicate values?
df.duplicated().sum() # In our case it is 0

0

In [38]:
# Que7 : How is the correlation between cols?
df.corr().round(3)
# Here we can see that S1 and S3 has high correlation
# Same as S2 and S4 has high correlation
# Also we can see that S1 and S3 has high negative correlation with S2 and S4

,time,x,y,S1,S2,S3,S4
time,1.000,0.000,0.0,-0.155,0.051,0.043,-0.303
x,0.000,1.000,0.0,-0.906,0.948,-0.931,0.623
y,0.000,0.000,1.0,-0.000,-0.000,-0.000,-0.000
S1,-0.155,-0.906,-0.0,1.000,-0.948,0.848,-0.396
S2,0.051,0.948,-0.0,-0.948,1.000,-0.964,0.651
S3,0.043,-0.931,-0.0,0.848,-0.964,1.000,-0.771
S4,-0.303,0.623,-0.0,-0.396,0.651,-0.771,1.000


In [39]:
# Adding new column to dataset 'M1 = e^-2.5x' a monotonic gradient along the x-axis

import numpy as np
df['M1'] = np.exp(-2.5 * df['x'])

In [40]:
# The time difference at every time step is 0.08 sec, so we can use dt = 0.08 sec up to last observation
dt = df['time'].unique()[1] - df['time'].unique()[0]

# We are calculating the gene expression change rate dSi_dt = (Si @ (X_previous, Y_previous) - Si @ (X_current, Y_current)) / time, where i = 1,2,3,4
for i in range(1, 5):
    gene_col = f'S{i}'
    df[f'dS{i}_dt'] = df.groupby(['x', 'y'])[gene_col].diff()/dt

# Here we used .diff() that calculate the difference between current value and previous value, but at the time t = 0 sec there is No previous values, so the dS{i}_dt become NaN and we should remove it.

df = df.dropna()

In [46]:
from sklearn.linear_model import LinearRegression

# We are using these columns as a features for linear regression, so that we can find the coefficients
X = df[['S1', 'S2', 'S3', 'S4', 'M1']]

# Matrix A (4X4) + Alpha regulation : This matrix shows the effect of one gene to other genes, weather one is prohibited, inhibited or not interacted for others
# Matrix A also include the value of Self-Regulation value which is diagonal of matrix A
genesEffectOnGenes = []

# Matrix B (4X2) : This matrix shows the effect of morphogens on intercellular genes, same as the matrix A it gives the mathematical idea that weather one is prohibited, inhibited or not interacted for genes
morphogensEffectsOnGenes = []

# we are using this equation y = w1x1 + w2x2 + w3x3 + w4x4 + w5x5 to find the weights (w1, w2, w3, w4, w5)
# Note: consider that we were not used b (intercept) term in above equation because we are assuming that the above equation line pass through origin (baseline)
for i in range(1, 5):
    y = df[f'dS{i}_dt']
    model = LinearRegression(fit_intercept=False)
    model.fit(X, y)
    coeffs = model.coef_

    genesEffectOnGenes.append(coeffs[:4])
    morphogensEffectsOnGenes.append([coeffs[4], 0]) # M2's effect is zero

# After training we get the w1...w4 as of A and w5 as of B

In [47]:
# First time we were chosen the 0.2 threshold because it's a clear separation was observed between numerical noise(<0.05) and dominant regulatory interactions(>0.4), but when we see that effect of M1 on S3 in matrix B was around 0.195 which is higher than noise then after we were chosen 0.18
Threshold = 0.18

# Now we are converting these matrices A and B in form of +1, -1 and 0, so that we can use it at the time of experiment B for calculate gene expression change rate (dSi_dt)

genesEffectOnGenes = np.array(genesEffectOnGenes)
# Removing the diagonal values because it is Regulatory term (Alpha)
np.fill_diagonal(genesEffectOnGenes, 0)
A = np.zeros_like(genesEffectOnGenes)
A[genesEffectOnGenes > Threshold] = 1
A[genesEffectOnGenes < -Threshold] = -1

#Making 4X2 Matrix B
morphogensEffectsOnGenes = np.array(morphogensEffectsOnGenes)
B = np.zeros_like(morphogensEffectsOnGenes)
B[morphogensEffectsOnGenes > Threshold] = 1
B[morphogensEffectsOnGenes < -Threshold] = -1

# At the time of steady state the Morphogens canceling out each other's effects so the effects of M1 and M2 on genes are opposite
B[0][1] = -B[0][0]
B[1][1] = -B[1][0]
B[2][1] = -B[2][0]

In [49]:
# Loading submission template
df_template = pd.read_csv('GRN_experiment_M1_removal_template.csv')

alpha = 1.02 # Alpha's value taken from genesEffectOnGenes's diagonal
B2 = B[:, 1] # M2's effects

# Setup Grid and Time
times = np.sort(df_template['time'].unique())
dt = times[1] - times[0]
unique_coords = df_template[['x', 'y']].drop_duplicates().values

# Initial State: t=0 and S=0 (Steady State)
S_state = np.zeros((len(unique_coords), 4))
M2_vals = np.exp(-((unique_coords[:, 0] - 0.5)**2 + (unique_coords[:, 1] - 0.5)**2) / (2 * (0.18)**2)) # M2's expression

results_storage = {}

# Calculating dSi_dt by temporal derivative approximation and storing them according to X-Y plan and time
for t in times:

    for i, (x, y) in enumerate(unique_coords):
        results_storage[(round(t, 4), round(x, 4), round(y, 4))] = S_state[i].copy()
    dS_dt = -alpha * S_state + S_state @ A.T + np.outer(M2_vals, B2)
    S_state += dS_dt * dt

# Final Data Re-ordering (Exactly like the template)
S_results = []

for _, row in df_template.iterrows():
    key = (round(row['time'], 4), round(row['x'], 4), round(row['y'], 4))
    S_results.append(results_storage[key])

S_results = np.array(S_results)

# Adding S1...S4
df_template['S1'] = S_results[:, 0]
df_template['S2'] = S_results[:, 1]
df_template['S3'] = S_results[:, 2]
df_template['S4'] = S_results[:, 3]

In [50]:
# Finally exporting our experiment B's csv file
df_template.to_csv('GRN_experiment_M1_removal.csv', index=False, float_format='%.6f')

In [ ]:
# Saving the Matrix A and B in text file
import numpy as np

Matrix_A = np.array([
    [0, 1, 0, 0],
    [-1, 0, 1, 0],
    [0, -1, 0, 1],
    [0, 0, -1, 0]
])

Matrix_B = np.array([
    [1, -1],
    [-1, 1],
    [1, -1],
    [0, 0]
])


with open('matrices.txt', 'w') as f:
    f.write("Matrix A:\n")
    np.savetxt(f, Matrix_A, fmt='%d')
    f.write("\nMatrix B:\n")
    np.savetxt(f, Matrix_B, fmt='%d')

In [5]:
dfOutput = pd.read_csv('GRN_experiment_M1_removal.csv')

In [ ]:
# Now again making profile report of output data to compare the Heat map
outputProfileReport = ProfileReport(dfOutput)
outputProfileReport.to_file('outputProfileReport')

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  6.58it/s]
c:\Users\bhumi\AppData\Local\Programs\Python\Python311\Lib\site-packages\ydata_profiling\profile_report.py:386: UserWarning: Extension  not supported. For now we assume .html was intended. To remove this warning, please use .html or .json.
  warnings.warn(
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 108.08it/s]
